# Geração combinatória: subconjuntos e permutações em ordem lexicográfica — Tutorial

**Algoritmos e Estruturas de Dados II (COMP0498) — UFS — 2026.2**

## Objetivos

Ao final deste tutorial você será capaz de:

- Decidir em C se uma sequência precede outra na **ordem lexicográfica**, inclusive no caso do prefixo;
- Enumerar os $2^n$ subconjuntos de $\{1,\ldots,n\}$ por **máscara de bits**, em $O(1)$ de espaço adicional;
- Gerar subconjuntos por **backtracking** em ordem lexicográfica dos elementos — e **podar** subárvores inteiras;
- Gerar as $n!$ permutações por backtracking em ordem lexicográfica, e mostrar que a variante **por trocas** não respeita essa ordem;
- Calcular o **sucessor lexicográfico** de uma permutação em $O(n)$ e **medir** que o custo total cresce como $n!$.

> As células de código escrevem programas em C com `%%writefile` e depois compilam com `gcc`.
> Rode as células **na ordem**.

In [ ]:
# Verifique se o gcc está disponível no seu ambiente
!gcc --version | head -1

## 1. Ordem lexicográfica

Dadas duas sequências $a$ e $b$ sobre um conjunto ordenado, dizemos que $a \prec_{\text{lex}} b$ quando,
na **primeira posição em que diferem**, o elemento de $a$ é menor; ou quando $a$ é **prefixo próprio**
de $b$. É exatamente a ordem do dicionário.

Duas consequências que usaremos o tempo todo:

- $(1) \prec (1,2)$ — o prefixo vem antes de qualquer extensão sua;
- $(1,3) \prec (2)$ — basta a primeira posição decidir, o tamanho não importa.

Daí a regra que governa todos os geradores deste tutorial: **se a recursão visita o nó antes dos
filhos e gera os filhos em ordem crescente, a saída sai em ordem lexicográfica.**

In [ ]:
%%writefile compara_lex.c
#include <stdio.h>

/* Devolve -1 se a < b, 0 se a == b, 1 se a > b, na ordem lexicografica. */
int compara_lex(const int a[], int na, const int b[], int nb) {
    for (int i = 0; i < na && i < nb; i++) {
        if (a[i] < b[i]) return -1;         /* primeira posicao em que diferem */
        if (a[i] > b[i]) return  1;
    }
    if (na < nb) return -1;                 /* a e' prefixo proprio de b */
    if (na > nb) return  1;
    return 0;                               /* iguais */
}

int main(void) {
    int a[] = {1,3},   b[] = {2};
    int c[] = {1,2,3}, d[] = {1,3};
    int e[] = {1},     f[] = {1,2};
    int g[] = {2,3},   h[] = {1,9};
    int i[] = {1,2},   j[] = {1,2};

    printf("(1,3)   ? (2)     -> %d\n", compara_lex(a, 2, b, 1));
    printf("(1,2,3) ? (1,3)   -> %d\n", compara_lex(c, 3, d, 2));
    printf("(1)     ? (1,2)   -> %d\n", compara_lex(e, 1, f, 2));
    printf("(2,3)   ? (1,9)   -> %d\n", compara_lex(g, 2, h, 2));
    printf("(1,2)   ? (1,2)   -> %d\n", compara_lex(i, 2, j, 2));

    printf("resumo: %d %d %d %d %d\n",
           compara_lex(a,2,b,1), compara_lex(c,3,d,2), compara_lex(e,1,f,2),
           compara_lex(g,2,h,2), compara_lex(i,2,j,2));
    return 0;
}

In [ ]:
# Compila e executa: os três primeiros pares são crescentes, o quarto não, o quinto é empate
!gcc -Wall compara_lex.c -o compara_lex && ./compara_lex
!./compara_lex | grep -qF 'resumo: -1 -1 -1 1 0' \
  && echo OK || echo 'Verifique: esperava resumo: -1 -1 -1 1 0'

### Exercício 1 — a comparação sem atalho

Escreva `menor_lex`, que devolve $1$ quando $a \prec_{\text{lex}} b$ e $0$ caso contrário — sem chamar
`compara_lex`. O caso do **prefixo** é o que separa uma implementação correta de uma quase correta.

In [ ]:
%%writefile exercicio1.c
#include <stdio.h>

/* Exercicio 1: devolva 1 se a sequencia a (tamanho na) for lexicograficamente
   MENOR que b (tamanho nb), e 0 caso contrario. Nao chame compara_lex:
   escreva a comparacao direta. */

int menor_lex(const int a[], int na, const int b[], int nb) {
    /* TODO: percorra as posicoes comuns; na primeira em que a e b diferem,
             decida. Se nenhuma diferir, o menor e' o mais curto. */
    return 0;
}

int main(void) {
    int p[] = {1,2,3}, q[] = {1,3};
    int r[] = {2},     s[] = {2,1};
    int t[] = {3,1},   u[] = {2,9};
    int x[] = {1,2},   y[] = {1,2};
    printf("menor_lex: %d%d%d%d\n",
           menor_lex(p,3,q,2),      /* 123 < 13      -> 1 */
           menor_lex(r,1,s,2),      /* 2 e' prefixo  -> 1 */
           menor_lex(t,2,u,2),      /* 31 > 29       -> 0 */
           menor_lex(x,2,y,2));     /* iguais        -> 0 */
    return 0;
}

In [ ]:
# Teste automático do Exercício 1
!gcc -Wall exercicio1.c -o exercicio1 && ./exercicio1
!./exercicio1 | grep -qF 'menor_lex: 1100' \
  && echo OK || echo 'Verifique sua implementacao: esperava menor_lex: 1100'

## 2. Subconjuntos por máscara de bits

Todo $S \subseteq \{1,\ldots,n\}$ corresponde a um **vetor característico** de $n$ bits, que por sua vez é
um inteiro em $[0, 2^n)$. Enumerar subconjuntos vira, então, **contar de $0$ a $2^n - 1$** — sem esquecer
nem repetir nenhum candidato.

Convenção usada aqui: o bit na posição $n-i$ representa o elemento $i$, de modo que o elemento $1$
fica no bit mais significativo.

Custo: $\Theta(n \cdot 2^n)$ de tempo e $O(1)$ de espaço adicional. Limite prático: $n \le 31$ com `unsigned`.
Os programas escrevem cada subconjunto entre **colchetes** — `[1,2]` — para que as células de teste
possam comparar a saída literalmente.

In [ ]:
%%writefile mascara.c
#include <stdio.h>

/* Imprime o subconjunto codificado na mascara m de n bits, entre colchetes.
   Convencao: o bit n-i representa o elemento i (o elemento 1 fica no bit mais alto). */
void imprime(unsigned m, int n) {
    int primeiro = 1;
    printf("[");
    for (int i = 1; i <= n; i++)
        if (m & (1u << (n - i))) {          /* o elemento i pertence a S? */
            if (!primeiro) printf(",");
            printf("%d", i);
            primeiro = 0;
        }
    printf("] ");
}

int main(void) {
    int n = 3;
    long total = 0;
    for (unsigned m = 0; m < (1u << n); m++) {   /* contar de 0 a 2^n - 1 */
        imprime(m, n);
        total++;
    }
    printf("\ntotal: %ld\n", total);
    return 0;
}

In [ ]:
# Compila e executa: 2^3 = 8 subconjuntos, na ordem em que a contagem binária os produz
!gcc -Wall mascara.c -o mascara && ./mascara
!./mascara | grep -qF '[] [3] [2] [2,3] [1] [1,3] [1,2] [1,2,3]' \
  && ./mascara | grep -qF 'total: 8' \
  && echo OK || echo 'Verifique: esperava os 8 subconjuntos na ordem da contagem binaria'

**Repare na ordem produzida:** `[] [3] [2] [2,3] [1] [1,3] [1,2] [1,2,3]`.

Ela é lexicográfica **sobre os vetores de bits** (`000 001 010 011 100 101 110 111`), mas **não** sobre
as listas crescentes de elementos — a ordem lexicográfica dos conjuntos seria
`[] [1] [1,2] [1,2,3] [1,3] [2] [2,3] [3]`. É exatamente essa diferença que a Seção 3 resolve.

### Exercício 2 — soma de subconjunto

Dado um vetor de inteiros positivos e um alvo $A$, conte **quantos** subconjuntos somam exatamente $A$.
Percorra as máscaras de $0$ a $2^n - 1$ — aqui a ordem não importa, só a garantia de que nenhum
candidato ficou de fora.

In [ ]:
%%writefile exercicio2.c
#include <stdio.h>

/* Exercicio 2: quantos subconjuntos de w[0..n-1] somam exatamente A?
   Use a mesma tecnica da Secao 2: percorra as mascaras de bits. */

int conta_subconjuntos(const int w[], int n, int A) {
    /* TODO: para cada mascara m de 0 a 2^n - 1, some os w[i] com o bit i ligado
             e conte quantas somas dao exatamente A. */
    return -1;
}

int main(void) {
    int w[] = {3, 34, 4, 12, 5, 2};
    printf("conta: %d %d %d\n",
           conta_subconjuntos(w, 6,  9),    /* {3,4,2} e {4,5}        -> 2 */
           conta_subconjuntos(w, 6, 30),    /* nenhum                 -> 0 */
           conta_subconjuntos(w, 6, 26));   /* todos menos o 34       -> 1 */
    return 0;
}

In [ ]:
# Teste automático do Exercício 2
!gcc -Wall exercicio2.c -o exercicio2 && ./exercicio2
!./exercicio2 | grep -qF 'conta: 2 0 1' \
  && echo OK || echo 'Verifique sua implementacao: esperava conta: 2 0 1'

## 3. Subconjuntos por backtracking em ordem lexicográfica

Agora construímos o subconjunto **incrementalmente**. Em cada nível escolhemos o próximo elemento
entre os **maiores que o último escolhido**, em ordem crescente, e visitamos o nó **ao chegar nele**:

```
visita o no atual
para i de inicio ate n:
    escolhe i
    recorre com inicio = i + 1
    desfaz a escolha
```

Pré-ordem com filhos crescentes $\Rightarrow$ ordem lexicográfica. Espaço $O(n)$ (a pilha da recursão).
A vantagem decisiva sobre a máscara é a **poda**: um `return` corta uma subárvore inteira, enquanto
o laço `for (m = 0; m < (1 << n); m++)` é obrigado a visitar as $2^n$ máscaras até o fim.

In [ ]:
%%writefile subconjuntos.c
#include <stdio.h>
#define MAXN 32

int n;
int sol[MAXN];      /* prefixo em construcao */
int k;              /* tamanho do prefixo    */
long total;

void visita(void) {
    for (int i = 0; i < k; i++) {
        printf(i ? ",%d" : "[%d", sol[i]);
    }
    if (k == 0) printf("[");
    printf("] ");
    total++;
}

void subconjuntos(int inicio) {
    visita();                               /* 1. visita o no' ao chegar nele */
    for (int i = inicio; i <= n; i++) {     /* 2. filhos em ordem crescente   */
        sol[k++] = i;                       /*    escolhe i                   */
        subconjuntos(i + 1);                /*    so' elementos maiores que i */
        k--;                                /*    desfaz a escolha            */
    }
}

int main(void) {
    n = 3; k = 0; total = 0;
    subconjuntos(1);
    printf("\ntotal: %ld\n", total);
    return 0;
}

In [ ]:
# Compila e executa: os mesmos 8 subconjuntos, agora em ordem lexicográfica
!gcc -Wall subconjuntos.c -o subconjuntos && ./subconjuntos
!echo; echo "mascara      :"; ./mascara | head -1
!echo "backtracking :"; ./subconjuntos | head -1
!./subconjuntos | grep -qF '[] [1] [1,2] [1,2,3] [1,3] [2] [2,3] [3]' \
  && echo OK || echo 'Verifique: esperava [] [1] [1,2] [1,2,3] [1,3] [2] [2,3] [3]'

### Exercício 3 — apenas os subconjuntos de tamanho $k$

Adapte o gerador para imprimir só os subconjuntos de tamanho **exatamente** `k_alvo`, mantendo a
ordem lexicográfica. Duas mudanças bastam: visitar apenas quando o prefixo já tem `k_alvo` elementos
e **parar de descer** nesse ponto — é a poda em ação. Para $n = 4$ e $k = 2$ devem sair
$\binom{4}{2} = 6$ subconjuntos.

In [ ]:
%%writefile exercicio3.c
#include <stdio.h>
#define MAXN 32

/* Exercicio 3: gere apenas os subconjuntos de tamanho EXATAMENTE k_alvo,
   mantendo a ordem lexicografica. */

int n, k_alvo;
int sol[MAXN], k;
long total;

void visita(void) {
    for (int i = 0; i < k; i++) printf(i ? ",%d" : "[%d", sol[i]);
    printf("] ");
    total++;
}

void k_subconjuntos(int inicio) {
    /* TODO: visite so' quando o prefixo tiver k_alvo elementos — e pare de descer
             nesse ponto (essa e' a poda). Caso contrario, estenda com os elementos
             de inicio ate n, em ordem crescente. */
}

int main(void) {
    n = 4; k_alvo = 2; k = 0; total = 0;
    k_subconjuntos(1);
    printf("\ntotal: %ld\n", total);
    return 0;
}

In [ ]:
# Teste automático do Exercício 3
!gcc -Wall exercicio3.c -o exercicio3 && ./exercicio3
!./exercicio3 | grep -qF '[1,2] [1,3] [1,4] [2,3] [2,4] [3,4]' \
  && ./exercicio3 | grep -qF 'total: 6' \
  && echo OK || echo 'Verifique sua implementacao: esperava [1,2] [1,3] [1,4] [2,3] [2,4] [3,4]'

## 4. Permutações por backtracking em ordem lexicográfica

Nas permutações **todos** os elementos entram; o que muda é a posição. Preenchemos as posições
$0, 1, \ldots, n-1$ e, em cada posição, testamos os valores ainda livres **em ordem crescente**,
marcando-os em `usado[]`.

Diferença estrutural em relação aos subconjuntos: só as **folhas** são visitadas — são $n!$ folhas,
todas na profundidade $n$. Custo $\Theta(n \cdot n!)$, espaço $O(n)$.

In [ ]:
%%writefile permutacoes.c
#include <stdio.h>
#include <string.h>
#define MAXN 16

int n;
int sol[MAXN];          /* permutacao parcial           */
int usado[MAXN + 1];    /* usado[v] = 1 se v ja' entrou */
long total;

void visita(void) {
    for (int i = 0; i < n; i++) printf("%d", sol[i]);
    printf(" ");
    total++;
}

void permutacoes(int pos) {
    if (pos == n) { visita(); return; }     /* folha: permutacao completa */
    for (int v = 1; v <= n; v++) {          /* valores em ordem crescente */
        if (usado[v]) continue;
        usado[v] = 1; sol[pos] = v;
        permutacoes(pos + 1);
        usado[v] = 0;                       /* desfaz */
    }
}

int main(void) {
    n = 3; total = 0;
    memset(usado, 0, sizeof usado);
    permutacoes(0);
    printf("\ntotal: %ld\n", total);
    return 0;
}

In [ ]:
# Compila e executa: 3! = 6 permutações, em ordem lexicográfica
!gcc -Wall permutacoes.c -o permutacoes && ./permutacoes
!./permutacoes | grep -qF '123 132 213 231 312 321' \
  && ./permutacoes | grep -qF 'total: 6' \
  && echo OK || echo 'Verifique: esperava 123 132 213 231 312 321'

### A variante por trocas gera as mesmas permutações — em outra ordem

A versão clássica **por trocas** dispensa o vetor `usado[]` e é mais rápida por um fator constante.
Em compensação, a troca **desordena o sufixo**: os valores que ainda restam deixam de estar em
ordem crescente, e a saída não é mais lexicográfica. Compare com `123 132 213 231 312 321`.

In [ ]:
%%writefile perm_troca.c
#include <stdio.h>

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

void permuta_troca(int v[], int pos, int n) {
    if (pos == n) {
        for (int i = 0; i < n; i++) printf("%d", v[i]);
        printf(" ");
        return;
    }
    for (int i = pos; i < n; i++) {
        troca(&v[pos], &v[i]);              /* poe v[i] na posicao pos */
        permuta_troca(v, pos + 1, n);
        troca(&v[pos], &v[i]);              /* desfaz */
    }
}

int main(void) {
    int v[] = {1, 2, 3};
    permuta_troca(v, 0, 3);
    printf("\n");
    return 0;
}

In [ ]:
# Compila e executa: mesmas 6 permutações, mas 321 aparece ANTES de 312
!gcc -Wall perm_troca.c -o perm_troca && ./perm_troca
!./perm_troca | grep -qF '123 132 213 231 321 312' \
  && echo OK || echo 'Verifique: esperava 123 132 213 231 321 312'

### Exercício 4 — as 24 permutações de $\{1,2,3,4\}$

Implemente `permutacoes` por backtracking com o vetor `usado[]`. A saída deve começar em `1234`,
terminar em `4321` e sair em ordem lexicográfica.

In [ ]:
%%writefile exercicio4.c
#include <stdio.h>
#include <string.h>
#define MAXN 16

/* Exercicio 4: as 24 permutacoes de {1,2,3,4} em ordem lexicografica. */

int n, sol[MAXN], usado[MAXN + 1];
long total;

void visita(void) {
    for (int i = 0; i < n; i++) printf("%d", sol[i]);
    printf(" ");
    total++;
}

void permutacoes(int pos) {
    /* TODO: se pos == n, a permutacao esta completa: visite e volte.
             Senao, teste os valores de 1 a n em ordem crescente, pulando os
             que ja' estao em usado[], e nao esqueca de desfazer a marcacao. */
}

int main(void) {
    n = 4; total = 0;
    memset(usado, 0, sizeof usado);
    permutacoes(0);
    printf("\ntotal: %ld\n", total);
    return 0;
}

In [ ]:
# Teste automático do Exercício 4
!gcc -Wall exercicio4.c -o exercicio4 && ./exercicio4
!./exercicio4 | grep -qF '1234 1243 1324 1342 1423 1432 2134' \
  && ./exercicio4 | grep -qF '4213 4231 4312 4321' \
  && ./exercicio4 | grep -qF 'total: 24' \
  && echo OK || echo 'Verifique sua implementacao: esperava 24 permutacoes de 1234 a 4321'

## 5. Próxima permutação em $O(n)$

Em vez de recursão, calculamos **in loco** a menor permutação estritamente maior que a atual — o
sucessor lexicográfico. É o algoritmo por trás de `std::next_permutation`.

Quatro passos, sobre `1 3 5 4 2`:

1. varrendo da direita, ache o **pivô**: o último $i$ com $v_i < v_{i+1}$ $\to$ pivô $= 3$ (o sufixo `5 4 2` é decrescente);
2. ache o elemento mais à direita **maior que o pivô** $\to$ `4`;
3. **troque** os dois $\to$ `1 4 5 3 2`;
4. **inverta** o sufixo à direita do pivô $\to$ `1 4 2 3 5`.

Se não existe pivô, a permutação era a última (toda decrescente). Custo $O(n)$ por chamada, mas
**$O(1)$ amortizado**: o pivô está na última posição em metade das chamadas, na penúltima em $1/6$
delas, e a média das comparações converge para $e \approx 2{,}7$.

In [ ]:
%%writefile proxima.c
#include <stdio.h>

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

static void inverte(int v[], int i, int j) {
    while (i < j) { troca(&v[i], &v[j]); i++; j--; }
}

/* Transforma v na MENOR permutacao estritamente maior que v.
   Devolve 1 se avancou e 0 se v ja' era a ultima (toda decrescente). */
int proxima_permutacao(int v[], int n) {
    int i = n - 2;
    while (i >= 0 && v[i] >= v[i + 1]) i--;      /* 1. pivo */
    if (i < 0) return 0;
    int j = n - 1;
    while (v[j] <= v[i]) j--;                    /* 2. sucessor do pivo no sufixo */
    troca(&v[i], &v[j]);                         /* 3. troca */
    inverte(v, i + 1, n - 1);                    /* 4. inverte o sufixo */
    return 1;
}

int main(void) {
    int w[] = {1, 3, 5, 4, 2};                   /* o exemplo dos slides */
    proxima_permutacao(w, 5);
    printf("sucessor de 13542: ");
    for (int i = 0; i < 5; i++) printf("%d", w[i]);
    printf("\n");

    int v[] = {1, 2, 3, 4};                      /* enumera a partir da 1a permutacao */
    long total = 0;
    do {
        for (int i = 0; i < 4; i++) printf("%d", v[i]);
        printf(" ");
        total++;
    } while (proxima_permutacao(v, 4));
    printf("\ntotal: %ld\n", total);
    return 0;
}

In [ ]:
# Compila e executa: o exemplo dos slides e a enumeração completa para n = 4
!gcc -Wall proxima.c -o proxima && ./proxima
!./proxima | grep -qF 'sucessor de 13542: 14235' \
  && ./proxima | grep -qF 'total: 24' \
  && echo OK || echo 'Verifique: esperava sucessor de 13542: 14235 e total: 24'

### Medindo o crescimento fatorial

Se o gerador é $O(1)$ amortizado por permutação, o tempo de enumerar tudo deve acompanhar $n!$ —
e não $n \cdot n!$. Rode a célula abaixo (a última linha leva cerca de um segundo) e olhe a coluna
`razao`: ela deve ficar perto de $n$, porque $n! = n \cdot (n-1)!$.

In [ ]:
%%writefile tempo_perm.c
#include <stdio.h>
#include <time.h>

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }
static void inverte(int v[], int i, int j) {
    while (i < j) { troca(&v[i], &v[j]); i++; j--; }
}

int proxima_permutacao(int v[], int n) {
    int i = n - 2;
    while (i >= 0 && v[i] >= v[i + 1]) i--;
    if (i < 0) return 0;
    int j = n - 1;
    while (v[j] <= v[i]) j--;
    troca(&v[i], &v[j]);
    inverte(v, i + 1, n - 1);
    return 1;
}

int main(void) {
    double anterior = 0.0;
    printf(" n    permutacoes   tempo(s)   razao\n");
    for (int n = 8; n <= 12; n++) {
        int v[16];
        volatile long long soma = 0;             /* impede o compilador de apagar o laco */
        long long cont = 0;
        for (int i = 0; i < n; i++) v[i] = i + 1;

        clock_t t0 = clock();
        do { cont++; soma += v[0]; } while (proxima_permutacao(v, n));
        double seg = (double) (clock() - t0) / CLOCKS_PER_SEC;

        printf("%2d %13lld %10.3f", n, cont, seg);
        if (anterior > 0.0) printf("   %5.1fx", seg / anterior);
        printf("\n");
        anterior = seg;
    }
    return 0;
}

In [ ]:
# Compila com -O2 e executa: cada linha custa cerca de n vezes a anterior
!gcc -Wall -O2 tempo_perm.c -o tempo_perm && ./tempo_perm

**Para responder:** olhando a tabela, quanto tempo levaria $n = 15$? E $n = 20$?
(Multiplique o tempo de $n = 12$ por $13 \times 14 \times 15$ e continue.) Não rode — estime.

### Exercício 5 — o sucessor lexicográfico

Implemente `proxima_permutacao` seguindo os quatro passos. Ela devolve $1$ se avançou e $0$ se `v`
já era a última permutação. Um erro comum é usar `<` no lugar de `<=` no passo 2 — teste com uma
permutação que tenha o sufixo com valores repetidos ao redor do pivô e veja o que acontece.

In [ ]:
%%writefile exercicio5.c
#include <stdio.h>

/* Exercicio 5: o sucessor lexicografico in loco. */

void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

void inverte(int v[], int i, int j) {
    while (i < j) { troca(&v[i], &v[j]); i++; j--; }
}

int proxima_permutacao(int v[], int n) {
    /* TODO: os quatro passos.
             1. i = n-2; recue enquanto v[i] >= v[i+1]   (acha o pivo)
             2. se i < 0, era a ultima: devolva 0
             3. j = n-1; recue enquanto v[j] <= v[i]; troque v[i] com v[j]
             4. inverta o sufixo v[i+1..n-1] e devolva 1 */
    return 0;
}

int main(void) {
    int w[] = {1, 3, 5, 4, 2};
    proxima_permutacao(w, 5);
    printf("sucessor de 13542: ");
    for (int i = 0; i < 5; i++) printf("%d", w[i]);
    printf("\n");

    int v[] = {1, 2, 3, 4};
    long total = 0;
    do {
        for (int i = 0; i < 4; i++) printf("%d", v[i]);
        printf(" ");
        total++;
    } while (proxima_permutacao(v, 4));
    printf("\ntotal: %ld\n", total);
    return 0;
}

In [ ]:
# Teste automático do Exercício 5
!gcc -Wall exercicio5.c -o exercicio5 && ./exercicio5
!./exercicio5 | grep -qF 'sucessor de 13542: 14235' \
  && ./exercicio5 | grep -qF '1234 1243 1324 1342' \
  && ./exercicio5 | grep -qF 'total: 24' \
  && echo OK || echo 'Verifique sua implementacao: esperava 14235 e as 24 permutacoes'

### Para pensar (responda em uma célula de texto)

1. `proxima_permutacao` é $O(n)$ no pior caso e $O(1)$ amortizado. Para $n = 4$, em quantas das $23$
   chamadas bem-sucedidas o laço do pivô dá **exatamente um passo** (isto é, o pivô é $v_{n-2}$)?
   Confira sua conta com a saída do Exercício 5.
2. O gerador de subconjuntos por backtracking visita $2^n$ nós e gasta $O(n)$ em cada um **só para
   imprimir**. Se ele apenas contasse os subconjuntos, qual seria o custo total? E o da máscara de
   bits na mesma tarefa? O que muda no expoente e o que muda apenas no fator.
3. A geração por trocas produz as mesmas $n!$ permutações da geração por backtracking, em outra
   ordem. Dê um problema em que essa diferença de ordem é irrelevante e outro em que ela torna o
   programa errado.

## Desafio Final — caixeiro viajante em ordem lexicográfica

Um entregador parte da cidade `0`, visita todas as outras exatamente uma vez e volta à origem.
A matriz `d[i][j]` dá a distância entre as cidades. Fixando a origem, sobram $(n-1)!$ rotas.

Sua tarefa: enumerar todas elas **em ordem lexicográfica** com `proxima_permutacao`, avaliar o custo
de cada uma e imprimir a melhor. Em caso de empate, a resposta deve ser a rota lexicograficamente
menor — e isso sai **de graça** se você só substituir a melhor quando o custo for **estritamente**
menor, porque a primeira rota ótima encontrada é a primeira na ordem lex.

Com a matriz de 5 cidades fornecida, a resposta é custo `32` na rota `0 2 1 3 4 0`, após avaliar
$4! = 24$ rotas.

**Extensão (opcional):** resolva o mesmo problema por backtracking com poda — abandone o prefixo
assim que o custo parcial já alcançar o melhor custo completo conhecido. Compare o número de nós
visitados com as 24 rotas da força bruta e repita para $n = 9$.

In [ ]:
%%writefile desafio.c
#include <stdio.h>
#include <limits.h>

#define N 5

int d[N][N] = {
    {  0, 12, 10, 19,  8 },
    { 12,  0,  3,  7,  2 },
    { 10,  3,  0,  6, 20 },
    { 19,  7,  6,  0,  4 },
    {  8,  2, 20,  4,  0 }
};

void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }
void inverte(int v[], int i, int j) {
    while (i < j) { troca(&v[i], &v[j]); i++; j--; }
}

int proxima_permutacao(int v[], int n) {
    /* TODO: reaproveite a sua implementacao do Exercicio 5. */
    return 0;
}

/* Custo da rota fechada 0 -> rota[0] -> ... -> rota[m-1] -> 0 */
int custo_rota(const int rota[], int m) {
    /* TODO: some d[0][rota[0]] + ... + d[rota[m-2]][rota[m-1]] + d[rota[m-1]][0]. */
    return 0;
}

int main(void) {
    int rota[N - 1] = {1, 2, 3, 4};       /* a menor permutacao das cidades != 0 */
    int melhor[N - 1], melhor_custo = INT_MAX;
    long avaliadas = 0;

    /* TODO: percorra as (N-1)! permutacoes de rota[] com proxima_permutacao,
             avalie cada uma com custo_rota e guarde a melhor. Troque a melhor
             so' quando o custo for ESTRITAMENTE menor: assim o empate fica com
             a rota que apareceu primeiro — a lexicograficamente menor. */

    printf("rotas avaliadas: %ld\n", avaliadas);
    printf("melhor custo: %d\n", melhor_custo);
    printf("rota: 0");
    for (int i = 0; i < N - 1; i++) printf(" %d", melhor[i]);
    printf(" 0\n");
    return 0;
}

In [ ]:
# Compile e rode o seu desafio
!gcc -Wall desafio.c -o desafio && ./desafio
!./desafio | grep -qF 'rotas avaliadas: 24' \
  && ./desafio | grep -qF 'melhor custo: 32' \
  && ./desafio | grep -qF 'rota: 0 2 1 3 4 0' \
  && echo OK || echo 'Verifique sua implementacao: esperava custo 32 na rota 0 2 1 3 4 0'

## Referências

Onde cada algoritmo deste tutorial aparece na literatura:

| Algoritmo | Referência e local |
|---|---|
| Máscara de bits (Seção 2) | Knuth, *TAOCP* 4A, seç. 7.2.1.1 (geração de todas as $n$-uplas); Kreher & Stinson, seç. 2.2 |
| Subconjuntos por backtracking (Seção 3) | Skiena, *The Algorithm Design Manual*, seç. 9.2.1; Ziviani, seç. 2.3 (tentativa e erro); Kreher & Stinson, cap. 4 |
| Subconjuntos de tamanho $k$ (Exercício 3) | Kreher & Stinson, seç. 2.3; Knuth, seç. 7.2.1.3 |
| Poda / soma de subconjunto (Exercício 2) | Skiena, seç. 9.3 (*search pruning*); Kreher & Stinson, seç. 4.6 (*bounding functions*); Cormen et al., seç. 35.5 |
| Permutações por backtracking (Seção 4) | Skiena, seç. 9.2.2; Kreher & Stinson, seç. 2.4 |
| Permutações por trocas (Seção 4) | Sedgewick (1977), *Permutation Generation Methods*; Knuth, seç. 7.2.1.2, Algoritmo P |
| `proxima_permutacao` (Seção 5) | Knuth, seç. 7.2.1.2, Algoritmo L; Kreher & Stinson, seç. 2.4. É a mesma rotina do `std::next_permutation` do C++ |
| Contagem de $2^n$ e $n!$ | Cormen et al., Apêndice C.1 |
| Caixeiro viajante (Desafio Final) | Cormen et al., seç. 35.2; Skiena, cap. 9 |

O sucessor lexicográfico (Algoritmo L) é atribuído por Knuth a Narayana Pandita, na Índia do
século XIV. Os dados bibliográficos completos estão em `../referencias.bib`.
